# 🎯 [K09_M06] 답변 형식 통일하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K09/k09_m06_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·상담 (Level 2) / K09 부대 정보 검색 시스템 |
| **미션** | 미션 06. 답변 형식 통일하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 프롬프트 엔지니어링 및 팩트체크 검증 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k09_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K09", "../uploads/K09"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k09_m01~m10) 준비 완료!")



---

## 🎯 미션 06. 답변 형식 통일하기
**소요시간**: 약 1.5시간

![미션 6 Few-shot 모범 예시 기반 5단 표준 출력 서식 제어 다이어그램](/uploads/common/K09_m06_format_fewshot.png)

### 📋 미션 개요 및 배경
질문자의 다양한 말투나 축약어에도 흔들리지 않고 규격화된 답변을 제공하도록 출력 포맷을 일원화합니다. '답변/위치/운영시간/문의처/근거' 템플릿과 Few-shot(예시)을 프롬프트에 포함하고, 반말·줄임말·오타 질문에서도 지정된 형식이 준수되는지 스트레스 테스트를 진행합니다. 사용자 경험(UX)의 일관성을 보장하고 정보 가독성을 극대화하는 출력 제어 기법을 익힙니다.

### 🎯 미션 학습목표
'답 / 위치 / 시간 / 문의 / 근거' 표준 출력 형식을 정의하고 예시 1개(Few-shot)를 프롬프트에 반영한 뒤, 반말·줄임말·오타 질문에서도 지정된 형식이 유지되는지 점검합니다. **이를 통해 사용자 입력의 다양성에도 일관된 규격으로 응답을 강제하는 출력 포맷 제어 기법과 프롬프트 견고성(Robustness) 검증 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 답변 형식(답·위치·시간·문의·근거)을 정한다.
#### 이 단계의 역할
모든 답변이 일관된 시각적 구조를 유지하도록 [답변 요약 / 위치 / 운영 시간 / 문의처 / 근거] 5단 표준 출력 서식을 설계합니다.

#### 해야 할 일

<li>장병들이 모바일이나 PC 화면에서 한눈에 정보를 파악할 수 있는 5개 필수 항목 서식을 정의합니다.

</li><li>각 항목별 줄바꿈, 구분 기호, 글머리 기호 규칙을 표준화합니다.

</li><li>프롬프트의 [출력 서식 지정] 섹션에 해당 템플릿을 명시합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[5단 표준 답변 서식 템플릿 설계]:** "[출력 포맷 규칙]: 모든 질문에 대해 반드시 아래 서식을 엄격히 준수하여 응답할 것:
📌 요약 답변: [한 문장으로 핵심 답변]
📍 시설 위치: [건물명 및 층수]
⏰ 운영 시간: [정규 시간 및 예외 시간]
📞 문의 연락: [내선 번호 또는 담당자]
🔍 인용 근거: [참조한 자료표의 시설명]"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **5대 항목 완전성:** 답변, 위치, 시간, 문의, 근거 5대 항목이 누락 없이 정의되었는가?
- **시각적 일관성:** 글머리 기호와 줄바꿈이 표준화되어 있는가?

#### 핵심 확인 지표
<blockquote>5단 표준 답변 출력 서식 명세서

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>답변 형식(필수 산출물)

</li>

#### 🔹 Step 2. 예시 1개를 넣는다.
#### 이 단계의 역할
AI가 출력 서식을 정확히 모방하도록 이상적인 질의응답 샘플 1쌍을 프롬프트에 주입하는 Few-shot 프롬프팅을 적용합니다.

#### 해야 할 일

<li>'의무실 진료'에 대한 모범 질의와 5단 서식이 완벽히 적용된 모범 응답 예시(Shot) 1건을 작성합니다.

</li><li>프롬프트 본문의 지시사항 바로 아래에 [질의응답 예시] 블록으로 삽입합니다.

</li><li>다른 시설 질문(사이버지식정보방 등)을 입력하여 예시의 형식을 완벽하게 계승하는지 확인합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[Few-shot 예시 주입 프롬프트]:** "[질의응답 Few-shot 예시]:
질문: 의무실 진료 언제까지 하나요?
답변:
📌 요약 답변: 평일 17:30까지 일반 진료가 가능합니다.
📍 시설 위치: 본관 1층 의무실
⏰ 운영 시간: 평일 09:00~17:30 (점심 소독 12:00~13:30 제외, 야간은 당직 접수)
📞 문의 연락: 내선 103 (의무대)
🔍 인용 근거: [의무실 운영 안내]"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **예시 모방도:** 예시로 제시한 글머리 기호, 줄바꿈, 어투(합니다 체)를 AI가 정확히 모방하는가?
- **형식 붕괴 방지:** 새로운 질문에 대해서도 줄바꿈이 흐트러지지 않는가?

#### 핵심 확인 지표
<blockquote>1건의 Few-shot 예시가 포함된 서식 고정 프롬프트

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>예시를 넣은 프롬프트(필수 산출물)

</li>

#### 🔹 Step 3. 반말·줄임말·오타 질문으로 시험한다.
#### 이 단계의 역할
사용자가 반말("사지방 몇시까지임?"), 군 축약어("체단실 어디?"), 오타("의무실 위치좀요오")로 거칠게 질문해도 정중하고 규격화된 서식을 유지하는지 스트레스 테스트를 수행합니다.

#### 해야 할 일

<li>정상 질문 3건을 반말, 줄임말, 자음 탈락/오타 질문으로 변형한 테스트 세트를 만듭니다.

</li><li>변형된 질문들을 순차적으로 프롬프트에 입력하고 출력 결과를 수집합니다.

</li><li>질문의 어투에 휘말리지 않고 항상 군 표준 존댓말과 5단 고정 서식을 준수하는지 판정표를 작성합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[비정형 입력 스트레스 테스트 질의]:** "[변형 테스트 질문 1]: 야 사지방 몇시까지 하냐?
[변형 테스트 질문 2]: 체단실 위치 어딧는지 알려줭 ㅠㅠ
[변형 테스트 질문 3]: 츙성마트 주말에도 염?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **어조의 일관성:** 사용자 질문이 반말이라도 AI는 군 표준 존댓말로 예의 바르게 응답하는가?
- **포맷 붕괴 방지:** 오타나 은어 질문에도 5단 서식이 깨지지 않고 유지되는가?

#### 핵심 확인 지표
<blockquote>변형 질문 3종 스트레스 테스트 결과표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>표현을 바꾼 질문 시험 결과(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
모바일 화면에서 한눈에 볼 수 있도록 핵심 정보만 2줄로 축약하는 모바일 전용 초간결 서식 옵션을 설계합니다.

#### 해야 할 일
'한 줄 요약' 칸을 맨 위에 추가해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[모바일 초간결 요약 서식 질의]:** "[모바일 간결 서식 규칙]: 스마트폰 화면용 초간결 모드 요청 시, [시설명 | 위치 | 운영시간]을 한 줄 표 형태로 압축 출력할 것."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **압축률 및 가독성:** 작은 모바일 화면에서도 스크롤 없이 핵심 정보가 읽히는가?

#### 핵심 확인 지표
<blockquote>모바일 초간결 서식 변환 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 서식 최적화 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M06] 답변 형식 통일하기 실습 가이드 및 작성 서식

## 1. 5단 표준 답변 포맷 명세서
- 장병들이 모바일과 PC 화면에서 정보를 직관적으로 스캔할 수 있도록 일원화된 5개 고정 항목입니다.

```markdown
📌 요약 답변: [한 문장으로 결론 제시]
📍 시설 위치: [건물명 및 층수/호수]
⏰ 운영 시간: [정규 운영 시간 및 예외 조건]
📞 문의 연락: [담당 부서 및 내선 번호]
🔍 인용 근거: [참조한 자료표의 시설명]
```

---

## 2. Few-shot 모범 예시 탑재 프롬프트

```markdown
[지시사항]: 아래 질의응답 예시의 서식과 어조를 100% 동일하게 모방하여 응답하라.

[모범 예시]:
질문: 의무실 진료 언제까지 하나요?
답변:
📌 요약 답변: 평일 17:30까지 일반 진료가 가능합니다.
📍 시설 위치: 본관 1층 서편 의무실
⏰ 운영 시간: 평일 09:00~17:30 (점심 소독 12:00~13:30 제외, 야간은 당직 접수)
📞 문의 연락: 내선 103 (의무대)
🔍 인용 근거: [대대 의무실 운영 안내]
```

---

## 3. 비정형 입력(반말·줄임말·오타) 스트레스 테스트 결과표

| 테스트 유형 | 입력 질문 | AI 출력 결과 (서식 준수 및 어조 검증) | 판정 |
| :--- | :--- | :--- | :---: |
| **반말 질문** | "야 사지방 몇시까지 하냐?" | 📌 요약 답변: 사이버지식정보방은 평일 21:00까지 이용하실 수 있습니다.<br>📍 시설 위치: 복지회관 2층 201호... (군 표준 경어체 및 5단 서식 준수) | **PASS** |
| **줄임말 질문** | "체단실 위치랑 샤워장 정보 좀" | 📌 요약 답변: 체력단련실은 본관 지하 1층에 위치하며 샤워장을 갖추고 있습니다.<br>⏰ 운영 시간: 평일 06:30~21:00... | **PASS** |
| **오타 질문** | "츙성마트 주말에도 문 염?" | 📌 요약 답변: 충성마트(PX)는 토요일 10:00~18:00까지 운영하며 일요일은 휴무입니다.<br>📍 시설 위치: 복지회관 1층... | **PASS** |

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **5단 표준 답변 포맷 명세서:** [📌 요약 답변 / 📍 위치 / ⏰ 운영 시간 / 📞 문의 / 🔍 근거] 템플릿
- **Few-shot 모범 질의응답 예시문 (의무실 안내 샘플):** AI의 어조와 글머리 기호, 줄바꿈을 100% 통제하는 모범 샷(Shot)
- **[학습 가이드] Few-shot 프롬프팅과 비정형 입력 스트레스 테스트:** 사용자 질문이 반말, 줄임말, 오타여도 규격화된 정중한 답변을 강제하는 프롬프트 견고성(Robustness) 검증법

**실습 팁:** Few-shot 예시에 들어가는 이모지(📌, 📍, ⏰ 등)와 들여쓰기 형태를 그대로 유지하라고 지시문에 한 줄 명시하면 포맷 통일성이 비약적으로 상승합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 출력 형식 지정 — Few-shot 예시를 활용하여 5단 표준 서식을 구축하고 비정형(반말·줄임말·오타) 질문에서도 일관성을 유지했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>5단 표준 답변 포맷 명세서 정의 완료

</li><li>모범 질의응답 예시가 포함된 Few-shot 프롬프트 완성

</li><li>반말, 줄임말, 오타 질문 3종 스트레스 테스트 전수 통과

</li>

#### 채점자 판정 메모
<blockquote>사용자가 거칠게 반말로 질문해도 AI가 품격 있는 군 경어체와 5단 규격 서식을 칼같이 지켜내는지 확인한다.

</blockquote>





In [ ]:
# [미션 06. 답변 형식 통일하기] 학생 실습 프롬프트 및 결과 기록란
mission_06_prompt = """
# [TODO: 미션 06] 5단 표준 포맷(요약, 위치, 시간, 연락, 근거)을 적용한 Few-shot 프롬프트를 작성하세요.
# 힌트: 사용자가 한눈에 정보를 파악할 수 있도록 고정된 5대 항목 템플릿과 모범 예시를 제공합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_06_result = """
### 📋 [미션 06 실습 결과]
## 1. 5단 표준 포맷 적용 질의응답 결과
- **질문**: 주말에 사지방에서 인터넷 강의 들으려는데 몇 시까지 하고 어디로 가면 되나요?
- **AI 출력 결과**:
  - 📌 요약 답변:
  - 📍 시설 위치:
  - ⏰ 운영 시간:
  - 📞 문의 연락:
  - 🔍 인용 근거:
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 06. 답변 형식 통일하기", mission_06_prompt, mission_06_result)

